# 🚀 Spaceship Titanic — Prediksi Transportasi ke Dimensi Lain

> **Kompetisi:** Spaceship Titanic (Kaggle)  
> **Tujuan:** Memprediksi apakah seorang penumpang **`Transported`** (terlempar ke dimensi alternatif) saat kapal bertabrakan dengan anomali ruang-waktu.  
> **Model Utama:** 🐱 **CatBoost Classifier**  
> **Teknik:** Feature Engineering + Rule-Based Imputation + 5-Fold Stratified Cross-Validation + Optuna Tuning  

---

## 📋 Daftar Isi
1. [Import Library](#1)
2. [Load Dataset](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Feature Engineering](#4)
5. [Smart Imputation](#5)
6. [Preprocessing & Persiapan Data](#6)
7. [Training dengan Cross-Validation](#7)
8. [Hyperparameter Tuning dengan Optuna](#8)
9. [Evaluasi Model Final](#9)
10. [Generate Submission File](#10)


---
<a id='1'></a>
## 📦 1. Import Library

Kita import semua library yang diperlukan:
- **`pandas`** & **`numpy`** — Manipulasi dan komputasi data
- **`matplotlib`** & **`seaborn`** — Visualisasi EDA dan evaluasi
- **`catboost`** — Model utama: CatBoostClassifier dengan dukungan native fitur kategorikal
- **`sklearn`** — Preprocessing, cross-validation, dan metrik evaluasi
- **`optuna`** — Hyperparameter tuning otomatis dengan Bayesian Optimization (TPE Sampler)


In [ ]:
# ── Core ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisasi ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine Learning ─────────────────────────────────────────
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# ── Hyperparameter Tuning ────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Styling ──────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

SEED = 42
np.random.seed(SEED)

print('✅ Semua library berhasil diimport!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
import catboost; print(f'   catboost: {catboost.__version__}')
print(f'   optuna  : {optuna.__version__}')

---
<a id='2'></a>
## 📂 2. Load Dataset

Dataset Spaceship Titanic terdiri dari 3 file:

| File | Keterangan |
|------|------------|
| `train.csv` | **8.693 penumpang** dengan label `Transported` — digunakan untuk training |
| `test.csv` | **4.277 penumpang** tanpa label — yang kita prediksi |
| `sample_submission.csv` | Format file submission yang diterima Kaggle |

### Deskripsi Kolom
| Kolom | Tipe | Deskripsi |
|---|---|---|
| `PassengerId` | String | Format `gggg_pp` — `gggg` adalah ID grup, `pp` nomor urut |
| `HomePlanet` | Kategorikal | Planet asal penumpang (Europa, Earth, Mars) |
| `CryoSleep` | Boolean | Apakah penumpang dalam kondisi beku (cryo) selama perjalanan |
| `Cabin` | String | Format `Deck/Num/Side` — lokasi kabin di kapal |
| `Destination` | Kategorikal | Planet tujuan (TRAPPIST-1e, 55 Cancri e, PSO J318.5-22) |
| `Age` | Numerik | Usia penumpang |
| `VIP` | Boolean | Apakah penumpang membayar layanan VIP |
| `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck` | Numerik | Tagihan fasilitas kapal |
| `Name` | String | Nama lengkap penumpang |
| `Transported` | Boolean | **Target** — apakah penumpang terlempar ke dimensi lain |


In [ ]:
# Path dataset — sesuaikan jika direktori berbeda
TRAIN_PATH  = '../Dataset/train.csv'
TEST_PATH   = '../Dataset/test.csv'
SAMPLE_PATH = '../Dataset/sample_submission.csv'

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
sample    = pd.read_csv(SAMPLE_PATH)

print('─' * 50)
print(f'  Train  shape : {train_raw.shape[0]:,} baris × {train_raw.shape[1]} kolom')
print(f'  Test   shape : {test_raw.shape[0]:,} baris × {test_raw.shape[1]} kolom')
print(f'  Sample shape : {sample.shape[0]:,} baris × {sample.shape[1]} kolom')
print('─' * 50)
print('\n5 Baris Pertama Train:')
train_raw.head(5)

---
<a id='3'></a>
## 🔍 3. Exploratory Data Analysis (EDA)

Sebelum membangun model, kita perlu **memahami data** terlebih dahulu:
- Distribusi target `Transported` (apakah seimbang?)
- Missing values di setiap kolom
- Distribusi fitur numerik per kelas target
- Hubungan fitur kategorikal dengan target

> 💡 **Dataset ini sangat seimbang (~50/50)**, sehingga **Accuracy** adalah metrik evaluasi yang tepat.


In [ ]:
# ── Ringkasan Missing Values ─────────────────────────────────
print('═' * 55)
print('  MISSING VALUES — TRAIN')
print('═' * 55)
missing     = train_raw.isnull().sum()
missing_pct = (missing / len(train_raw) * 100).round(2)
missing_df  = pd.DataFrame({'Count': missing, 'Percentage (%)': missing_pct})
print(missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False))
print('\n  Catatan: Semua kolom kecuali PassengerId, Name,'
      ' dan Transported punya missing values (~2-3%)')

In [ ]:
# ── Distribusi Target ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = train_raw['Transported'].value_counts()
colors = ['#4ECDC4', '#FF6B6B']

# Pie Chart
axes[0].pie(
    target_counts.values,
    labels=['Transported (True)', 'Not Transported (False)'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Distribusi Target — Transported', fontweight='bold')

# Count Plot
train_plot = train_raw.copy()
train_plot['Transported'] = train_plot['Transported'].astype(str)
sns.countplot(data=train_plot, x='Transported',
              palette={'True': '#4ECDC4', 'False': '#FF6B6B'}, ax=axes[1])
axes[1].set_title('Count per Kelas', fontweight='bold')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 30,
                f'{int(bar.get_height()):,}', ha='center', fontweight='bold')

plt.suptitle('🎯 Target Distribution — Spaceship Titanic', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f'Total  : {len(train_raw):,} penumpang')
print(f'True   : {target_counts[True]:,} ({target_counts[True]/len(train_raw)*100:.2f}%)')
print(f'False  : {target_counts[False]:,} ({target_counts[False]/len(train_raw)*100:.2f}%)')
print('✅ Dataset seimbang — Accuracy cocok sebagai metrik utama.')

In [ ]:
# ── Distribusi Fitur Numerik per Kelas Target ────────────────
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for transported, grp in train_raw.groupby('Transported'):
        label = 'Transported' if transported else 'Not Transported'
        color = '#4ECDC4' if transported else '#FF6B6B'
        axes[i].hist(grp[col].dropna(), bins=40, alpha=0.6,
                    label=label, color=color, edgecolor='none')
    axes[i].set_title(f'Distribusi {col}', fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('📊 Distribusi Fitur Numerik per Kelas Target', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Fitur Kategorikal vs Target ──────────────────────────────
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    cross = pd.crosstab(train_raw[col].astype(str),
                       train_raw['Transported'].astype(str),
                       normalize='index') * 100
    cross.plot(kind='bar', ax=axes[i], color=['#FF6B6B', '#4ECDC4'],
              edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'{col} vs Transported (%)', fontweight='bold')
    axes[i].set_ylabel('Persentase (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(['Not Transported', 'Transported'], fontsize=8)
    axes[i].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('📊 Fitur Kategorikal vs Target', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n💡 Insight Kunci:')
print('   - CryoSleep=True  → Hampir 80% penumpang Transported!')
print('   - Europa          → Lebih sedikit yang Transported dibanding Earth.')
print('   - VIP=True        → Sedikit lebih banyak yang Not Transported.')
print('   - Destination     → TRAPPIST-1e paling banyak punya Transported.')

---
<a id='4'></a>
## ⚙️ 4. Feature Engineering

Ini adalah bagian **paling krusial** dalam kompetisi ini. Kita akan mengekstrak informasi tersembunyi dari fitur-fitur yang ada:

| Fitur Baru | Sumber | Alasan / Insight |
|---|---|---|
| `Cabin_Deck`, `Cabin_Num`, `Cabin_Side` | `Cabin` (format `D/N/S`) | Lokasi spasial kabin saat benturan berpengaruh |
| `Group_Id`, `Pax_Num`, `Group_Size`, `Is_Solo` | `PassengerId` (format `gggg_pp`) | Kelompok perjalanan — anggota grup sering punya nasib serupa |
| `Last_Name` | `Name` | Anggota keluarga (nama belakang sama) sering Transported bersama |
| `Total_Spent` | Sum 5 biaya fasilitas | Proksi kuat: jika spent > 0, pasti **tidak** CryoSleep |
| `Zero_Spent` | `Total_Spent == 0` | Berkorelasi kuat dengan CryoSleep dan Transported |
| `Log_*` | Log transform pengeluaran | Mengurangi skewness distribusi nilai pengeluaran |
| `Ratio_*` | Proporsi tiap fasilitas | Pola preferensi fasilitas per penumpang |
| `Age_Group` | Binning `Age` | Anak-anak dan lansia punya pola berbeda |


In [ ]:
def feature_engineering(df):
    """Fungsi utama feature engineering — diterapkan ke train & test."""
    df = df.copy()

    # ── 1. Dekomposisi PassengerId ──────────────────────────
    df['Group_Id']   = df['PassengerId'].str.split('_').str[0]
    df['Pax_Num']    = df['PassengerId'].str.split('_').str[1].astype(int)
    group_sizes      = df.groupby('Group_Id')['PassengerId'].transform('count')
    df['Group_Size'] = group_sizes
    df['Is_Solo']    = (df['Group_Size'] == 1).astype(int)

    # ── 2. Dekomposisi Cabin ────────────────────────────────
    cabin_split      = df['Cabin'].str.split('/', expand=True)
    df['Cabin_Deck'] = cabin_split[0]
    df['Cabin_Num']  = pd.to_numeric(cabin_split[1], errors='coerce')
    df['Cabin_Side'] = cabin_split[2]

    # ── 3. Ekstrak Last Name ────────────────────────────────
    df['Last_Name']  = df['Name'].str.split().str[-1]

    # ── 4. Total & Status Pengeluaran ───────────────────────
    spend_cols       = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['Total_Spent'] = df[spend_cols].sum(axis=1, skipna=True)
    df['Zero_Spent']  = (df['Total_Spent'] == 0).astype(int)

    # Log transform untuk mengurangi skewness
    for col in spend_cols:
        df[f'Log_{col}'] = np.log1p(df[col].fillna(0))
    df['Log_Total_Spent'] = np.log1p(df['Total_Spent'])

    # Ratio pengeluaran per fasilitas
    for col in spend_cols:
        df[f'Ratio_{col}'] = df[col] / (df['Total_Spent'] + 1)

    # ── 5. Age Group (Binning) ──────────────────────────────
    df['Age_Group'] = pd.cut(
        df['Age'],
        bins=[0, 12, 17, 25, 40, 60, 200],
        labels=['Child', 'Teen', 'Young_Adult', 'Adult', 'Middle_Age', 'Senior'],
        right=True
    ).astype(str)

    return df


train = feature_engineering(train_raw)
test  = feature_engineering(test_raw)

new_cols = [c for c in train.columns if c not in train_raw.columns]
print(f'✅ Feature engineering selesai!')
print(f'   Train : {train_raw.shape[1]} → {train.shape[1]} kolom (+{len(new_cols)} fitur baru)')
print(f'   Test  : {test_raw.shape[1]} → {test.shape[1]} kolom')
print(f'\n📌 Fitur Baru: {new_cols}')

---
<a id='5'></a>
## 🧠 5. Smart Imputation (Imputasi Berbasis Aturan)

Kita **tidak** menggunakan imputasi statistik biasa (mean/median). Kita memanfaatkan **logika domain** dari universe kompetisi ini:

```
Aturan 1: Jika Total_Spent > 0  → CryoSleep pasti False
           (Penumpang dalam cryo tidak bisa menggunakan fasilitas)

Aturan 2: Jika CryoSleep = True → Semua biaya fasilitas = 0
           (Penumpang beku tidak mungkin berbelanja)

Aturan 3: HomePlanet biasanya sama untuk 1 Group_Id atau Last_Name yang sama
           (Anggota keluarga/grup berangkat dari planet yang sama)

Aturan 4: Cabin_Deck/Side sering sama dalam 1 Group
           (Anggota grup sering ditempatkan di kabin berdekatan)
```

> 🎯 **Strategi**: Rule-based dulu, fallback ke median/mode untuk sisa missing values.


In [ ]:
def smart_impute(df):
    """Imputasi cerdas berbasis aturan domain Spaceship Titanic."""
    df = df.copy()
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

    # ── Aturan 1: Jika spent > 0, CryoSleep pasti False ─────
    mask_spent = df['Total_Spent'] > 0
    df.loc[mask_spent & df['CryoSleep'].isna(), 'CryoSleep'] = False

    # ── Aturan 2: Jika CryoSleep True, semua pengeluaran = 0 ─
    mask_cryo = (df['CryoSleep'] == True) | (df['CryoSleep'] == 'True')
    for col in spend_cols:
        df.loc[mask_cryo & df[col].isna(), col] = 0.0

    # ── Aturan 3: HomePlanet dari Group ID ──────────────────
    planet_map = df.groupby('Group_Id')['HomePlanet'].agg(
        lambda x: x.mode().iloc[0] if x.notna().any() else np.nan
    )
    df['HomePlanet'] = df['HomePlanet'].fillna(df['Group_Id'].map(planet_map))

    # ── Aturan 4: HomePlanet dari Last Name ─────────────────
    planet_lastname = df.groupby('Last_Name')['HomePlanet'].agg(
        lambda x: x.mode().iloc[0] if x.notna().any() else np.nan
    )
    df['HomePlanet'] = df['HomePlanet'].fillna(df['Last_Name'].map(planet_lastname))

    # ── Aturan 5: Cabin dari Group ───────────────────────────
    for cabin_part in ['Cabin_Deck', 'Cabin_Side']:
        cabin_map = df.groupby('Group_Id')[cabin_part].agg(
            lambda x: x.mode().iloc[0] if x.notna().any() else np.nan
        )
        df[cabin_part] = df[cabin_part].fillna(df['Group_Id'].map(cabin_map))

    # ── Fallback Median/Mode ─────────────────────────────────
    num_cols_fill = ['Age', 'Cabin_Num'] + spend_cols
    for col in num_cols_fill:
        df[col] = df[col].fillna(df[col].median())

    cat_cols_fill = ['CryoSleep', 'HomePlanet', 'Destination', 'VIP',
                     'Cabin_Deck', 'Cabin_Side', 'Age_Group']
    for col in cat_cols_fill:
        df[col] = df[col].fillna(df[col].mode()[0])

    # ── Update fitur turunan setelah imputasi ────────────────
    df['Total_Spent']     = df[spend_cols].sum(axis=1)
    df['Zero_Spent']      = (df['Total_Spent'] == 0).astype(int)
    df['Log_Total_Spent'] = np.log1p(df['Total_Spent'])
    for col in spend_cols:
        df[f'Log_{col}']   = np.log1p(df[col])
        df[f'Ratio_{col}'] = df[col] / (df['Total_Spent'] + 1)

    return df


train = smart_impute(train)
test  = smart_impute(test)

print('✅ Smart Imputation selesai!')
print(f'   Missing values tersisa di train : {train.isnull().sum().sum()}')
print(f'   Missing values tersisa di test  : {test.isnull().sum().sum()}')

---
<a id='6'></a>
## 🔧 6. Preprocessing & Persiapan Data

**CatBoost** adalah salah satu keunggulan besar karena ia menangani fitur kategorikal secara **native** tanpa perlu One-Hot Encoding atau Label Encoding manual — ini menghemat preprocessing dan mencegah *information loss*.

Kita hanya perlu:
1. Mendefinisikan daftar **FEATURES** yang akan digunakan
2. Menentukan mana saja **CAT_FEATURES** (CatBoost perlu tahu indeks/nama kolom kategorikal)
3. Memisahkan `X`, `y`, dan `X_test`


In [ ]:
# ── Definisi Fitur ───────────────────────────────────────────
FEATURES = [
    # Fitur Original
    'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
    'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',

    # Fitur dari Cabin
    'Cabin_Deck', 'Cabin_Num', 'Cabin_Side',

    # Fitur dari Group
    'Group_Size', 'Is_Solo', 'Pax_Num',

    # Fitur Pengeluaran
    'Total_Spent', 'Zero_Spent', 'Log_Total_Spent',
    'Log_RoomService', 'Log_FoodCourt', 'Log_ShoppingMall', 'Log_Spa', 'Log_VRDeck',
    'Ratio_RoomService', 'Ratio_FoodCourt', 'Ratio_ShoppingMall', 'Ratio_Spa', 'Ratio_VRDeck',

    # Fitur Demografi
    'Age_Group',

    # Fitur Identitas Grup
    'Last_Name', 'Group_Id',
]

# Fitur kategorikal — CatBoost menangani ini secara native
CAT_FEATURES = [
    'HomePlanet', 'CryoSleep', 'Destination', 'VIP',
    'Cabin_Deck', 'Cabin_Side', 'Age_Group',
    'Last_Name', 'Group_Id'
]

TARGET = 'Transported'

# ── Persiapan X, y, X_test ───────────────────────────────────
X      = train[FEATURES].copy()
y      = train[TARGET].astype(int)
X_test = test[FEATURES].copy()

# CatBoost membutuhkan kolom kategorikal dalam format string
for col in CAT_FEATURES:
    X[col]      = X[col].astype(str).replace('nan', 'Unknown')
    X_test[col] = X_test[col].astype(str).replace('nan', 'Unknown')

print(f'✅ Data siap untuk training!')
print(f'   X shape      : {X.shape}  ({len(FEATURES)} fitur)')
print(f'   y shape      : {y.shape}')
print(f'   X_test shape : {X_test.shape}')
print(f'\n   Fitur Numerik     : {len(FEATURES) - len(CAT_FEATURES)}')
print(f'   Fitur Kategorikal : {len(CAT_FEATURES)}')
print(f'\n   Target distribution:')
print(f'   {dict(y.value_counts())}')

---
<a id='7'></a>
## 🏋️ 7. Training CatBoost dengan 5-Fold Stratified Cross-Validation

**Mengapa Cross-Validation?**
- Memberikan estimasi performa yang **lebih reliable** dibanding single train-val split
- Setiap data mendapat giliran sebagai data validasi (tidak ada data yang terbuang)
- Membantu deteksi overfitting

**Strategi Training:**
```
5-Fold Stratified CV:
├── Fold 1: [████░░░░░░] → Train 80% | Val 20%
├── Fold 2: [░░████░░░░] → Train 80% | Val 20%
├── Fold 3: [░░░░████░░] → Train 80% | Val 20%
├── Fold 4: [░░░░░░████] → Train 80% | Val 20%
└── Fold 5: [████░░░░░░] → Train 80% | Val 20%
                                              ↓
Test Prediction = Average dari 5 model (Ensembling)
```

**Early Stopping** (`od_wait=100`): Training berhenti otomatis jika tidak ada improvement selama 100 iterasi — mencegah overfitting.


In [ ]:
# ── Konfigurasi Cross-Validation ─────────────────────────────
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ── Parameter Baseline CatBoost ──────────────────────────────
CATBOOST_PARAMS = dict(
    iterations          = 2000,
    learning_rate       = 0.05,
    depth               = 6,
    l2_leaf_reg         = 3,
    bagging_temperature = 0.5,
    border_count        = 128,
    od_type             = 'Iter',
    od_wait             = 100,
    eval_metric         = 'Accuracy',
    random_seed         = SEED,
    verbose             = 0,
    cat_features        = CAT_FEATURES,
    allow_writing_files = False,
)

# ── Container Hasil ──────────────────────────────────────────
oof_preds   = np.zeros(len(X))       # Out-of-Fold probabilities
test_preds  = np.zeros(len(X_test))  # Test predictions (rata-rata 5 fold)
fold_scores = []                     # Accuracy per fold
models      = []                     # Simpan model tiap fold

print('🚀 Memulai 5-Fold Stratified Cross-Validation...\n')
print(f'{"Fold":<6} {"Train Size":<13} {"Val Size":<12} {"Val Accuracy":<16} {"Val AUC"}')
print('─' * 60)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Buat CatBoost Pool (format data native CatBoost)
    train_pool = Pool(X_tr, y_tr, cat_features=CAT_FEATURES)
    val_pool   = Pool(X_val, y_val, cat_features=CAT_FEATURES)
    test_pool  = Pool(X_test, cat_features=CAT_FEATURES)

    # Training
    model = CatBoostClassifier(**CATBOOST_PARAMS)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    # Prediksi probabilitas
    val_prob  = model.predict_proba(val_pool)[:, 1]
    val_pred  = (val_prob >= 0.5).astype(int)
    test_prob = model.predict_proba(test_pool)[:, 1]

    # Simpan hasil
    oof_preds[val_idx] = val_prob
    test_preds        += test_prob / N_FOLDS
    models.append(model)

    acc = accuracy_score(y_val, val_pred)
    auc = roc_auc_score(y_val, val_prob)
    fold_scores.append(acc)
    print(f'  {fold:<4} {len(X_tr):<13,} {len(X_val):<12,} {acc:<16.5f} {auc:.5f}')

print('─' * 60)
print(f'\n📊 OOF Accuracy (Baseline) : {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}')
print(f'📊 OOF ROC-AUC (Baseline)  : {roc_auc_score(y, oof_preds):.5f}')

---
<a id='8'></a>
## 🎯 8. Hyperparameter Tuning dengan Optuna

**Optuna** menggunakan **TPE Sampler (Tree-structured Parzen Estimator)** — sebuah metode Bayesian Optimization yang:
- Lebih cerdas dari Random Search (belajar dari trial sebelumnya)
- Jauh lebih cepat dari Grid Search (tidak exhaustive)
- Otomatis berhenti jika konvergen

**Parameter yang di-tune:**

| Parameter | Range | Dampak |
|---|---|---|
| `learning_rate` | [0.01, 0.2] | Kecepatan belajar — kecil = lambat tapi lebih akurat |
| `depth` | [4, 10] | Kedalaman pohon — dalam = kompleks, risiko overfit |
| `l2_leaf_reg` | [1, 10] | Regularisasi L2 — besar = lebih conservative |
| `bagging_temperature` | [0, 1] | Tingkat randomisasi sampling data |
| `border_count` | [32, 255] | Presisi pembagian fitur numerik |
| `random_strength` | [0.1, 10] | Randomisasi pada score penentuan split |


In [ ]:
def objective(trial):
    """Fungsi objektif Optuna — maximize OOF Accuracy (3-Fold untuk kecepatan)."""
    params = dict(
        iterations          = 1000,
        learning_rate       = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        depth               = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1, 10),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 1.0),
        border_count        = trial.suggest_int('border_count', 32, 255),
        random_strength     = trial.suggest_float('random_strength', 0.1, 10, log=True),
        od_type             = 'Iter',
        od_wait             = 50,
        eval_metric         = 'Accuracy',
        random_seed         = SEED,
        verbose             = 0,
        cat_features        = CAT_FEATURES,
        allow_writing_files = False,
    )

    scores = []
    cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    for train_idx, val_idx in cv3.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = CatBoostClassifier(**params)
        model.fit(
            Pool(X_tr, y_tr, cat_features=CAT_FEATURES),
            eval_set=Pool(X_val, y_val, cat_features=CAT_FEATURES),
            use_best_model=True
        )
        scores.append(accuracy_score(y_val, model.predict(X_val)))
    return np.mean(scores)


print('🔍 Memulai Optuna Hyperparameter Tuning (50 trials)...')
print('   Menggunakan 3-Fold CV untuk kecepatan optimasi.\n')

study = optuna.create_study(
    direction='maximize',
    study_name='catboost_spaceship_titanic',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n🏆 Best Trial #{study.best_trial.number}')
print(f'   Best Accuracy (3-Fold CV): {study.best_value:.5f}')
print(f'\n📌 Best Hyperparameters:')
for k, v in study.best_params.items():
    print(f'   {k:<25}: {v}')

In [ ]:
# ── Visualisasi Optuna ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trial_values = [t.value for t in study.trials]
best_so_far  = np.maximum.accumulate(trial_values)
axes[0].plot(trial_values, alpha=0.4, color='#4ECDC4', label='Trial Score', marker='o', markersize=3)
axes[0].plot(best_so_far, color='#FF6B6B', linewidth=2.5, label='Best So Far')
axes[0].set_title('Optuna — Optimization History', fontweight='bold')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Parameter importance
try:
    param_importances = optuna.importance.get_param_importances(study)
    params_sorted = dict(sorted(param_importances.items(), key=lambda x: x[1]))
    axes[1].barh(list(params_sorted.keys()), list(params_sorted.values()),
                color='#4ECDC4', edgecolor='white')
    axes[1].set_title('Hyperparameter Importance', fontweight='bold')
    axes[1].set_xlabel('Importance')
except Exception as e:
    axes[1].text(0.5, 0.5, f'Cannot compute importance:\n{e}',
                ha='center', va='center', transform=axes[1].transAxes)

plt.suptitle('🎯 Optuna Tuning Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Retrain dengan Best Hyperparameters (5-Fold CV) ─────────
BEST_PARAMS = dict(
    **study.best_params,
    iterations          = 3000,
    od_type             = 'Iter',
    od_wait             = 100,
    eval_metric         = 'Accuracy',
    random_seed         = SEED,
    verbose             = 0,
    cat_features        = CAT_FEATURES,
    allow_writing_files = False,
)

oof_preds_tuned   = np.zeros(len(X))
test_preds_tuned  = np.zeros(len(X_test))
fold_scores_tuned = []
models_tuned      = []

print('🚀 Retraining dengan Best Hyperparameters (5-Fold CV)...\n')
print(f'{"Fold":<6} {"Val Accuracy":<16} {"Val AUC"}')
print('─' * 35)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=CAT_FEATURES)
    val_pool   = Pool(X_val, y_val, cat_features=CAT_FEATURES)
    test_pool  = Pool(X_test, cat_features=CAT_FEATURES)

    model = CatBoostClassifier(**BEST_PARAMS)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    val_prob  = model.predict_proba(val_pool)[:, 1]
    val_pred  = (val_prob >= 0.5).astype(int)
    test_prob = model.predict_proba(test_pool)[:, 1]

    oof_preds_tuned[val_idx] = val_prob
    test_preds_tuned        += test_prob / N_FOLDS
    models_tuned.append(model)

    acc = accuracy_score(y_val, val_pred)
    auc = roc_auc_score(y_val, val_prob)
    fold_scores_tuned.append(acc)
    print(f'  {fold:<4} {acc:<16.5f} {auc:.5f}')

print('─' * 35)
print(f'\n📊 Baseline OOF Accuracy   : {np.mean(fold_scores):.5f}')
print(f'📊 Tuned OOF Accuracy      : {np.mean(fold_scores_tuned):.5f} ± {np.std(fold_scores_tuned):.5f}')
print(f'📊 Tuned OOF ROC-AUC       : {roc_auc_score(y, oof_preds_tuned):.5f}')
improvement = np.mean(fold_scores_tuned) - np.mean(fold_scores)
print(f'\n✅ Improvement dari Baseline: +{improvement:.5f}')

---
<a id='9'></a>
## 📈 9. Evaluasi Model Final

Evaluasi komprehensif menggunakan Out-of-Fold (OOF) predictions — prediksi yang dibuat saat data tidak digunakan untuk training (validasi murni, bebas overfitting):

- **Confusion Matrix** — Visualisasi True/False Positives & Negatives
- **Classification Report** — Precision, Recall, F1-Score per kelas
- **Feature Importance** — Fitur mana yang paling berpengaruh terhadap prediksi
- **Probability Distribution** — Seberapa percaya diri model dalam prediksinya


In [ ]:
# ── Ringkasan Performa Final ─────────────────────────────────
oof_pred_binary = (oof_preds_tuned >= 0.5).astype(int)

print('=' * 60)
print('              EVALUASI MODEL FINAL (OOF)')
print('=' * 60)
print(f'  Model             : CatBoost Classifier')
print(f'  Cross-Validation  : 5-Fold Stratified')
print(f'  Hyperparameters   : Optuna Tuned (50 trials)')
print(f'  Total Fitur       : {len(FEATURES)}')
print('─' * 60)
print(f'  OOF Accuracy      : {accuracy_score(y, oof_pred_binary):.5f}')
print(f'  OOF ROC-AUC       : {roc_auc_score(y, oof_preds_tuned):.5f}')
print('─' * 60)
print('\n  Classification Report:')
print(classification_report(
    y, oof_pred_binary,
    target_names=['Not Transported', 'Transported']
))
print('=' * 60)

In [ ]:
# ── Confusion Matrix & Fold Accuracy ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y, oof_pred_binary)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Transported', 'Transported']
)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (OOF)', fontweight='bold', fontsize=13)

# TP, TN, FP, FN
tn, fp, fn, tp = cm.ravel()
print(f'  True Positive  (benar prediksi Transported)    : {tp:,}')
print(f'  True Negative  (benar prediksi Not Transported): {tn:,}')
print(f'  False Positive (salah prediksi Transported)    : {fp:,}')
print(f'  False Negative (salah prediksi Not Transported): {fn:,}')

# Fold Accuracy Comparison
fold_nums = [f'Fold {i}' for i in range(1, N_FOLDS + 1)]
mean_acc  = np.mean(fold_scores_tuned)
bar_colors = ['#4ECDC4' if s >= mean_acc else '#FF6B6B' for s in fold_scores_tuned]
bars = axes[1].bar(fold_nums, fold_scores_tuned, color=bar_colors, edgecolor='white', linewidth=1.5)
axes[1].axhline(mean_acc, color='#FFD93D', linewidth=2.5,
                linestyle='--', label=f'Mean: {mean_acc:.4f}')
for bar, score in zip(bars, fold_scores_tuned):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                f'{score:.4f}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_ylim(min(fold_scores_tuned) - 0.01, max(fold_scores_tuned) + 0.02)
axes[1].set_title('Accuracy per Fold (Tuned)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('📈 Evaluasi Model — CatBoost (Spaceship Titanic)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance (Rata-rata dari 5 Fold) ───────────────
feat_imp = np.zeros(len(FEATURES))
for model in models_tuned:
    feat_imp += model.get_feature_importance() / N_FOLDS

feat_imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': feat_imp
}).sort_values('Importance', ascending=False)

TOP_N = 25
top_feats = feat_imp_df.head(TOP_N)

fig, ax = plt.subplots(figsize=(12, 10))
palette = ['#4ECDC4' if i < 5 else '#7EC8C8' if i < 15 else '#B0D9D9' for i in range(TOP_N)]
bars = ax.barh(
    top_feats['Feature'][::-1], top_feats['Importance'][::-1],
    color=palette[::-1], edgecolor='white', linewidth=0.5
)
for bar, val in zip(bars, top_feats['Importance'][::-1].values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)

ax.axvline(feat_imp_df['Importance'].mean(), color='#FF6B6B',
           linestyle='--', alpha=0.7, label='Mean Importance')
ax.set_title(f'🌟 Top {TOP_N} Feature Importance — CatBoost\n(Rata-rata 5 Fold)',
            fontsize=15, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.legend()
plt.tight_layout()
plt.show()

print('\n🔝 Top 10 Fitur Paling Berpengaruh:')
print(feat_imp_df.head(10).to_string(index=False))

---
<a id='10'></a>
## 📤 10. Generate File Submission

Langkah terakhir: menghasilkan file `submission_catboost.csv` sesuai format yang diterima Kaggle.

**Format yang diharapkan:**
```
PassengerId,Transported
0013_01,True
0018_01,False
...
```

> 💡 Prediksi final adalah **rata-rata probabilitas** dari 5 model (soft ensemble) — lebih stabil dari prediksi single model.


In [ ]:
# ── Probability Distribution Test ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# OOF Probability per kelas
for label, color in [(0, '#FF6B6B'), (1, '#4ECDC4')]:
    mask  = y == label
    name  = 'Not Transported' if label == 0 else 'Transported'
    axes[0].hist(oof_preds_tuned[mask], bins=50, alpha=0.65,
                color=color, label=name, edgecolor='none')
axes[0].axvline(0.5, color='white', linestyle='--', linewidth=2, label='Threshold = 0.5')
axes[0].set_title('OOF Probability Distribution (Train)', fontweight='bold')
axes[0].set_xlabel('Predicted Probability (Transported=1)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Test Probability
axes[1].hist(test_preds_tuned, bins=50, color='#4ECDC4', alpha=0.8, edgecolor='none')
axes[1].axvline(0.5, color='#FF6B6B', linestyle='--', linewidth=2.5, label='Threshold = 0.5')
axes[1].set_title('Test Set Probability Distribution', fontweight='bold')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('📊 Distribusi Probabilitas Prediksi', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Generate & Simpan Submission ─────────────────────────────
import os

# Prediksi final: True/False (boolean sesuai format Kaggle)
final_pred_binary = (test_preds_tuned >= 0.5)

submission = pd.DataFrame({
    'PassengerId': test_raw['PassengerId'],
    'Transported': final_pred_binary
})

# Simpan ke folder Submissions
os.makedirs('../Submissions', exist_ok=True)
SUBMISSION_PATH = '../Submissions/submission_catboost.csv'
submission.to_csv(SUBMISSION_PATH, index=False)

print('✅ File submission berhasil dibuat!')
print(f'📁 Lokasi  : {os.path.abspath(SUBMISSION_PATH)}')
print(f'📊 Total   : {len(submission):,} prediksi')
print()

# Distribusi prediksi
pred_dist = submission['Transported'].value_counts()
n_true  = int(pred_dist.get(True, 0))
n_false = int(pred_dist.get(False, 0))
print(f'Distribusi Prediksi:')
print(f'  Transported (True) : {n_true:,}  ({n_true/len(submission)*100:.1f}%)')
print(f'  Not Transported    : {n_false:,} ({n_false/len(submission)*100:.1f}%)')
print()
print('── Preview 10 Baris Pertama ──')
print(submission.head(10).to_string(index=False))

In [ ]:
# ── Validasi Format Submission ───────────────────────────────
sample_ids = set(sample['PassengerId'])
sub_ids    = set(submission['PassengerId'])

print('🔍 Validasi Format Submission...')
print(f'  ID di sample_submission : {len(sample_ids):,}')
print(f'  ID di submission kita   : {len(sub_ids):,}')
print(f'  ID yang cocok           : {len(sample_ids & sub_ids):,}')
print(f'  ID yang hilang          : {len(sample_ids - sub_ids):,}')
print(f'  Tipe Transported        : {submission["Transported"].dtype}')

if len(sample_ids) == len(sub_ids) == len(sample_ids & sub_ids):
    print('\n🎉 Submission VALID — Siap untuk diupload ke Kaggle!')
else:
    print('\n⚠️  Ada ketidaksesuaian ID — Periksa kembali!')

In [ ]:
# ── Ringkasan Final ──────────────────────────────────────────
print()
print('╔' + '═' * 58 + '╗')
print('║  🚀 RINGKASAN HASIL AKHIR — SPACESHIP TITANIC           ║')
print('╠' + '═' * 58 + '╣')
print(f'║  Model           : CatBoost Classifier                  ║')
print(f'║  Cross-Validation: 5-Fold Stratified                    ║')
print(f'║  Tuning          : Optuna TPE (50 trials)                ║')
print(f'║  Total Fitur     : {len(FEATURES):<39}║')
print('╠' + '═' * 58 + '╣')
oof_acc = np.mean(fold_scores_tuned)
oof_auc = roc_auc_score(y, oof_preds_tuned)
oof_std = np.std(fold_scores_tuned)
print(f'║  OOF Accuracy    : {oof_acc:.5f} ± {oof_std:.5f}              ║')
print(f'║  OOF ROC-AUC     : {oof_auc:.5f}                         ║')
print('╠' + '═' * 58 + '╣')
print(f'║  Output File     : submission_catboost.csv               ║')
print('╚' + '═' * 58 + '╝')
print()
print('🎉 Pipeline selesai! Siap untuk upload ke Kaggle!')
print('   Upload di: https://www.kaggle.com/competitions/spaceship-titanic/submissions')